# Range-Doppler Maps from an RGBD Sequence

Depth pixels are back-projected into world coordinates and become the scatter
sites of one `Radar.simulate` call per output frame.

**Pipeline:** depth frames -> `PointTargets(positions=..., rcs=..., trajectory=...)`
-> `Radar.simulate` -> `Frame.range_doppler`

No depth sequence ships with this repository. Point `sequence_path` at a
`(T, H, W)` depth array saved as `.npz` (for instance the output of
`python -m examples.preprocess_rfgen_rd --input-dir <recording>`). When that
file is absent, as in the saved outputs below, the notebook uses the example's
synthetic sequence: a wall at 4 m and a person-sized block in front of it walking
towards the camera at 1.2 m/s.

Each radar frame keeps the depth samples valid at both ends of the frame and
gives them a trajectory, their positions interpolated between neighbouring
source frames. The scene therefore moves within the frame, `Motion.auto()`
resolves to adaptive sampling, and a moving surface lands in its own Doppler
bins.

Site count is the cost driver: every site is a Channel endpoint in both legs.

In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

repo_root = pathlib.Path.cwd().parents[0]
sys.path.insert(0, str(repo_root))

from examples.rgbd_range_doppler import (
    DECIBEL_FLOOR,
    DEFAULT_RADAR_CONFIG,
    OUTPUT_FRAME_RATE_HZ,
    build_arg_parser,
    build_scene,
    build_site_sampler,
    load_rgbd_sequence,
    synthetic_sequence,
)
from witwin.radar import Radar

assert torch.cuda.is_available(), "the propagation solve is native CUDA only"

## 1. Input

In [ ]:
sequence_path = repo_root / "output" / "rfgen_rgbd_sequence.npz"
# The loaders read their knobs off the same argparse namespace the script uses,
# so a notebook run and a command-line run cannot drift apart.
if sequence_path.exists():
    args = build_arg_parser().parse_args(["--input", str(sequence_path), "--max-points", "4096"])
    sequence = load_rgbd_sequence(sequence_path, args)
    print("input: recorded sequence")
else:
    args = build_arg_parser().parse_args(["--synthetic"])
    sequence = synthetic_sequence()
    print("input: synthetic sequence (no recording at output/rfgen_rgbd_sequence.npz)")
print("depths", None if sequence.depths is None else sequence.depths.shape, "fps", sequence.fps)

## 2. Preview one depth frame

In [ ]:
if sequence.depths is not None:
    figure, ax = plt.subplots(figsize=(4, 3), dpi=72)
    picture = ax.imshow(sequence.depths[0], cmap="viridis")
    ax.set_title("Depth frame 0")
    ax.axis("off")
    figure.colorbar(picture, ax=ax, fraction=0.046, pad=0.04, label="m")
    plt.tight_layout()

## 3. Radar, world and site sampler

In [ ]:
radar = Radar.from_dict(DEFAULT_RADAR_CONFIG, device="cuda", position=(0.0, 0.0, 0.0), look_at=(0.0, 0.0, -1.0))
scene = build_scene()
sampler = build_site_sampler(sequence, args=args, device=radar.device)
# One cross section for every depth sample, in m^2.
site_rcs = float(args.site_rcs)
# One radar frame lasts chirps x transmitters x chirp period (TDM), in s.
frame_duration = radar.waveform.chirp_period * radar.num_tx * radar.waveform.chirps_per_frame
print("source frames", sampler.source_frames, "at", sampler.source_fps, "fps; span", round(sampler.total_time, 4), "s")
print("radar frame", round(frame_duration * 1e3, 2), "ms; sites at t=0:", tuple(sampler.sites(0.0).shape))

## 4. One radar frame

`result.axes` is the metadata every processing stage reads, built once by the
simulation from the waveform spec and the array the cube came from, and
`result.frame(0)` pairs one frame's cube with it. The `Frame` methods below are
facades over `witwin.radar.processing`. `window="hann"` tapers slow time; the
range stage stays rectangular, because the cube is already a range spectrum.

In [ ]:
targets = sampler.frame_targets(0.0, frame_duration, site_rcs)
result = radar.simulate(scene, targets, times=(0.0,), los=True, reflections=0)
axes = result.axes
rd = result.frame(0).range_doppler(window="hann")
print("cube", tuple(result.cube.shape), "rows", result.last_radar_paths.path_count, "sampling", result.motion_sampling)
print("velocity bin", round(axes.velocity_bin_mps, 4), "m/s; range bin", round(axes.range_bin_m, 4), "m")

In [ ]:
half = axes.range_bin_count // 2
ranges = axes.range_m[:half].cpu().numpy()
velocities = axes.velocity_mps.cpu().numpy()
rd_db = (20.0 * torch.log10(rd.data[0, 0].abs() + DECIBEL_FLOOR)).cpu().numpy()[:, :half]

moving = np.abs(velocities) > axes.velocity_bin_mps
doppler_bin, range_bin = np.unravel_index(int(rd_db[moving].argmax()), rd_db[moving].shape)
print(f"strongest moving cell: {ranges[range_bin]:.2f} m, {velocities[moving][doppler_bin]:+.3f} m/s")

figure, ax = plt.subplots(figsize=(6, 3), dpi=72)
picture = ax.imshow(
    rd_db, extent=[ranges[0], ranges[-1], velocities[0], velocities[-1]], origin="lower", aspect="auto", cmap="jet"
)
ax.set_xlabel("Range (m)")
ax.set_ylabel("Velocity (m/s)")
ax.set_title("RGBD Range-Doppler, frame 0")
figure.colorbar(picture, ax=ax, label="dB re 1 sqrt(W)")
plt.tight_layout()

## 5. A short sequence

In [ ]:
num_frames = 4
maps = []
for index in range(num_frames):
    # ``OUTPUT_FRAME_RATE_HZ`` is this example's own scheduling choice, not a
    # radar parameter: nothing inside the simulator repeats on a cadence.
    instant = index / OUTPUT_FRAME_RATE_HZ
    step = radar.simulate(
        scene, sampler.frame_targets(instant, instant + frame_duration, site_rcs), times=(instant,), reflections=0
    )
    cell = step.frame(0).range_doppler(window="hann").data[0, 0]
    maps.append((20.0 * torch.log10(cell.abs() + DECIBEL_FLOOR)).cpu().numpy()[:, :half])

stack = np.stack(maps, axis=0)
print("stack", stack.shape, "dB range", round(float(stack.min()), 1), round(float(stack.max()), 1))

figure, axesrow = plt.subplots(1, num_frames, figsize=(3 * num_frames, 2.6), dpi=72, sharey=True)
for index, ax in enumerate(axesrow):
    ax.imshow(
        stack[index],
        extent=[ranges[0], ranges[-1], velocities[0], velocities[-1]],
        origin="lower",
        aspect="auto",
        cmap="jet",
        vmin=float(stack.max()) - 60.0,
        vmax=float(stack.max()),
    )
    ax.set_title(f"frame {index}")
    ax.set_xlabel("Range (m)")
axesrow[0].set_ylabel("Velocity (m/s)")
plt.tight_layout()

The wall sits on the zero-velocity line at 4 m. The block sits at its walking
speed, +1.2 m/s (positive is closing), and moves in by 0.12 m from one frame to
the next, 0.1 s apart. With `--no-intra-frame-motion` (frozen sites, one
observation per frame) both would sit on the zero-velocity line and only the
range change would remain.